# Laser a Microring Ibrido III-V su Silicio con Controllo MOS

![chirp](img/chirp.png)

## Controllo Unico tramite MOS

Il punto fondamentale dell'architettura a tre terminali risiede nella capacità di isolare la sintonizzazione spettrale dai disturbi termici. Durante il normale funzionamento del dispositivo:

1. La corrente di iniezione (Terminali P1-N) viene impostata a un valore fisso e costante per garantire il guadagno ottico stazionario (regime ad onda continua, CW).

2. Il microring raggiunge un equilibrio termico statico che stabilizza il punto di lavoro (fissando un red-shift di base iniziale).

3. Da questo momento in poi, il controllo in frequenza e la sintonizzazione fine del laser vengono affidati unicamente al terminale MOS (P2-N) in tensione.

Variando la tensione sul MOS l'indice di rifrazione varia in frazioni di nanosecondo. Poiché la corrente del diodo non viene alterata e il MOS non dissipa potenza, il laser viene controllato ed eventualmente modulato ad altissima velocità senza chirp.

## Air Trench

Per massimizzare la velocità di risposta del terminale MOS e minimizzare i consumi si introduce uno spazio piano di aria nel silicio.

* Il condensatore MOS integrato nel ring crea intrinsecamente una capacità parassita. Senza accorgimenti, questa capacità rallenterebbe il circuito elettrico ($RC$), limitando la banda di sintonizzazione e modulazione del dispositivo.

* L'air Trench isola e confina geometricamente la regione del condensatore MOS esclusivamente attorno all'area sub-micrometrica in cui viaggia il modo ottico fondamentale della guida d'onda.

Grazie all'air trench si riesce a ridurre la capacità parassita del dispositivo da 10.5 pF a 1.23 pF. Questo abbattimento permette al MOS di rispondere quasi istantaneamente ai segnali elettrici, abilitando sintonizzazioni rapide con consumi energetici irrisori (efficienza fino a $6\text{ nm/nW}$).

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def simula_laser_ibrido_semplice(corrente_mA=0.0, tensione_MOS_V=0.0):

    fig, (ax_spec, ax_map) = plt.subplots(1, 2, figsize=(15, 6))
    plt.subplots_adjust(wspace=0.25)
    
    # Parametri fisici
    lambda_0 = 1554.650  # nm (lunghezza d'onda centrale)
    coeff_termico = 0.15  # nm/mA (Red-shift dovuto al calore)
    coeff_mos = -0.05     # nm/V (Blue-shift indotto dal MOS)
    
    # Calcolo degli shift
    shift_termico = corrente_mA * coeff_termico
    shift_mos = tensione_MOS_V * coeff_mos
    shift_totale = shift_termico + shift_mos
    lambda_attuale = lambda_0 + shift_totale

    # Grafico1: Spettro di Emissione del Laser

    wav = np.linspace(1554.0, 1555.3, 1000)
    
    # Fondo di rumore a -80 dBm, picco laser che sale fino a -10 dBm
    larghezza_riga = 0.015  # nm
    profilo = larghezza_riga**2 / ((wav - lambda_attuale)**2 + larghezza_riga**2)
    intensita = -80 + 70 * profilo  
    
    ax_spec.plot(wav, intensita, color='purple', linewidth=2.5, label="Spettro Ottico del Laser")
    ax_spec.axvline(lambda_0, color='darkgray', linestyle='--', alpha=0.8, label="$\lambda_0$ (Riferimento)")
    ax_spec.axvline(lambda_attuale, color='red', linestyle='-', linewidth=1.5, label="Picco Attuale")
    
    # Box
    textstr = f"Picco Lasing: {lambda_attuale:.3f} nm\nShift Netto: {shift_totale:+.3f} nm"
    props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='darkgray')
    ax_spec.text(0.05, 0.93, textstr, transform=ax_spec.transAxes, fontsize=10, bbox=props)
    
    ax_spec.set_title("Spettro di Emissione", fontweight='bold')
    ax_spec.set_xlabel("Wavelength (nm)")
    ax_spec.set_ylabel("Intensity (dBm)")
    ax_spec.set_ylim(-85, -5)  # Adatta scala per evidenziare il picco
    ax_spec.grid(True, linestyle=':', alpha=0.5)
    ax_spec.legend(loc="upper right")

    # Grafico2: Mappa di Controllo (Shift vs Input)
    x_axis = np.linspace(0, 5, 100)
    
    ax_map.plot(x_axis, x_axis * coeff_termico, color='crimson', linewidth=2, label="Effetto Termico Diodo (P1-N)")
    ax_map.plot(x_axis, x_axis * coeff_mos, color='dodgerblue', linewidth=2, label="Effetto Plasma MOS (P2-N)")
    ax_map.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    
    # Punti correnti determinati dagli slider
    ax_map.scatter(corrente_mA, shift_termico, color='crimson', s=130, zorder=5, edgecolor='black', label="Stato Corrente")
    ax_map.scatter(tensione_MOS_V, shift_mos, color='dodgerblue', s=130, zorder=5, edgecolor='black', label="Stato Tensione MOS")
    ax_map.scatter(max(corrente_mA, tensione_MOS_V), shift_totale, color='darkviolet', marker='X', s=180, zorder=6, label="Risultante Netta")
    
    ax_map.set_title("Shift Lunghezza d'Onda", fontweight='bold')
    ax_map.set_xlabel("Input di Controllo (mA Corrente / V Tensione MOS)")
    ax_map.set_ylabel("Lasing Wavelength Shift (nm)")
    ax_map.set_xlim(-0.2, 5.2)
    ax_map.set_ylim(-0.4, 0.4)
    ax_map.grid(True, linestyle=':', alpha=0.5)
    ax_map.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

# Slider
interact(simula_laser_ibrido_semplice, 
         corrente_mA=widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1, description="Corrente InP (mA):"),
         tensione_MOS_V=widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description="Tensione MOS (V):"));

<>:32: SyntaxWarning: invalid escape sequence '\l'
<>:32: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_9450/701580485.py:32: SyntaxWarning: invalid escape sequence '\l'
  ax_spec.axvline(lambda_0, color='darkgray', linestyle='--', alpha=0.8, label="$\lambda_0$ (Riferimento)")


interactive(children=(FloatSlider(value=0.0, description='Corrente InP (mA):', max=2.0), FloatSlider(value=0.0…

---

# Doppio Microring e Effetto Vernier

![image.png](img/vernier.png)

Nei laser integrati è fondamentale poter sintonizzare la lunghezza d'onda del laser in modo preciso e allargare il range di sintonizzazione senza incorrere in salti di modo incontrollati.

Un singolo microring risonatore agisce come un filtro ottico spettrale passabanda, caratterizzato da picchi di trasmissione periodici distanziati da un parametro chiamato Free Spectral Range (FSR), definito come:

$$FSR = \frac{\lambda^2}{n_g \cdot L}$$

Dove $\lambda$ è la lunghezza d'onda, $n_g$ è l'indice di rifrazione di gruppo della guida d'onda e $L$ è la circonferenza geometrica dell'anello. Per avere un filtro molto selettivo (alto fattore $Q$) l'anello deve essere relativamente grande, il che riduce l'FSR e limita il range in cui il laser può essere sintonizzato.

### La Soluzione: Due Microring in Serie

Per superare questo limite si usa una configurazione a doppio microring sfruttando l'Effetto Vernier:

* I due anelli sono progettati intenzionalmente con raggi leggermente differenti, in modo da avere due FSR distinti ma vicini ($FSR_1 \neq FSR_2$).

* La luce può circolare stabilmente nella cavità ed essere emessa come riga laser solo in corrispondenza delle lunghezze d'onda in cui i picchi di trasmissione di entrambi i microring si sovrappongono perfettamente.

* L'FSR effettivo del sistema combinato ($FSR_{tot}$) diventa molto più grande dei singoli FSR, estendendo enormemente il range di sintonizzazione del laser:

$$FSR_{tot} \approx \frac{FSR_1 \cdot FSR_2}{|FSR_1 - FSR_2|}$$

In [32]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

%matplotlib inline

# Spettro di frequenze Banda C
wavelengths = np.linspace(1530, 1565, 2000)

def simulate_vernier_effect(Tuning_Anello_2):
    
    # Parametri dell'Anello 1 (Fisso)
    fsr1 = 4.0        # Free Spectral Range dell'anello 1 (nm)
    lambda_ref1 = 1540.0

    # Risposta ottica dell'anello 1 (simulata con una funzione di sinc al quadrato)
    spettro_anello1 = 1 / (1 + (np.sin(np.pi * (wavelengths - lambda_ref1) / fsr1) / 0.08)**2)
    
    # Parametri dell'Anello 2 (Sintonizzabile tramite lo slider)
    fsr2 = 4.4        # FSR leggermente diverso per l'effetto Vernier (nm)

    # Risposta ottica anello 2
    lambda_ref2 = 1540.0 + Tuning_Anello_2  
    spettro_anello2 = 1 / (1 + (np.sin(np.pi * (wavelengths - lambda_ref2) / fsr2) / 0.08)**2)
    
    # Risposta Totale del Sistema Vernier
    spettro_vernier = spettro_anello1 * spettro_anello2
    
    # Grafici

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Grafico1: Spettro dei singoli anelli
    ax1.plot(wavelengths, spettro_anello1, color='dodgerblue', lw=1.5, label='Anello 1 (Fisso, $FSR_1 = 4.0$ nm)')
    ax1.plot(wavelengths, spettro_anello2, color='orange', lw=1.5, linestyle='--', label='Anello 2 (Riscaldato, $FSR_2 = 4.4$ nm)')
    ax1.set_title("Risposta Spettrale dei Singoli Microring", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Trasmissione", fontsize=11)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.legend(loc='upper right')
    
    # Grafico2: Risposta combinata (Effetto Vernier)
    ax2.plot(wavelengths, spettro_vernier, color='forestgreen', lw=2.5, label='Doppio Anello (Effetto Vernier)')       
    ax2.set_title("Risposta Combinata", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Wavelength (nm)", fontsize=11)
    ax2.set_ylabel("Trasmissione Netta", fontsize=11)
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Slider
interact(
    simulate_vernier_effect,
    Tuning_Anello_2=FloatSlider(
        value=0.0, min=-2.0, max=4.0, step=0.1, 
        description='Riscaldamento Anello 2 (Spostamento nm):', 
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )
);

interactive(children=(FloatSlider(value=0.0, description='Riscaldamento Anello 2 (Spostamento nm):', layout=La…

---

# Laser Ibridi Integrati ($InP + Si_3N_4$)

## 1. Assorbimento a Due Fotoni (TPA)

Il **TPA** è un effetto ottico che si manifesta nei materiali semiconduttori quando sono sottoposti a intensità luminose estremamente elevate.

* **Meccanismo Fisico:** In regime lineare, un materiale è trasparente se l'energia del singolo fotone è inferiore al suo bandgap ($h\nu < E_g$). Tuttavia, ad alte potenze la densità fotonica è così elevata che un elettrone può assorbire due fotoni quasi simultaneamente, accumulando l'energia necessaria ($2h\nu > E_g$) per saltare in banda di conduzione.

* **Il Limite del Silicio Classico ($Si$):** Il Silicio ha un bandgap stretto (~1.11 eV). La luce a 1550 nm (0.8 eV) non viene assorbita singolarmente, ma attiva pesantemente il TPA ad alta potenza. Il TPA genera portatori liberi (elettroni liberi) che causano l'Assorbimento da Portatori Liberi, il quale dissipa energia sotto forma di calore, altera l'indice di rifrazione (effetto termo-ottico) e allarga drasticamente la larghezza di riga (linewidth) del laser.

* **La Soluzione del Nitruro di Silicio ($Si_3N_4$):** Il Nitruro di Silicio ha invece un bandgap molto ampio (~5 eV). L'energia combinata di due fotoni a 1550 nm ($2 \times 0.8\text{ eV} = 1.6\text{ eV}$) è ampiamente inferiore al bandgap del materiale. Il $Si_3N_4$ è quindi migliore, permettendo al laser di operare a potenze elevate stabili (oltre i 100 mW) mantenendo la linea stretta.

### Come cambia il TPA

La soglia oltre la quale l'Assorbimento a Due Fotoni diventa distruttivo per il funzionamento del laser non è un valore fisso, ma varia in base a tre parametri ingegneristici:

* **Il Bandgap del Materiale:** Il TPA si verifica solo se l'energia di due fotoni combinati riesce a superare l'energia di bandgap del materiale.

* **L'Area Efficace della Guida d'Onda:** Il TPA non dipende dalla potenza totale in assoluto, ma dalla *densità* di potenza (quanti fotoni sono ammassati nello stesso spazio). Se la guida d'onda ha una sezione microscopica la luce si concentra e questo abbassa la soglia in milliwatt, facendo attivare il TPA anche a potenze molto basse. Guide d'onda più larghe aiutano a "diluire" la luce, alzando la soglia.

* **Il Fattore di Qualità $Q$ del Risonatore:** Nei microring l'intensità della luce viene amplificata perché i fotoni rimangono intrappolati a girare nell'anello per molto tempo. Più il fattore $Q$ del microring è elevato, più l'energia interna si accumula. Di conseguenza, un risonatore molto efficiente paradossalmente anticipa il problema, abbassando la soglia di potenza esterna necessaria a scatenare il TPA nel Silicio.

In [33]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

%matplotlib inline

def simulate_all_materials_and_geometry(Potenza_Scelta_mW, Soglia_TPA_Silicio_mW):

    # Parametri fissi
    L_gain_mm = 0.5  # Lunghezza dell InP
    linewidth_solitario = 2.0e6  # 2 MHz a 1 mW per il solo InP
    linewidth_floor = 20.0  # Pavimento tecnico minimo di 20 Hz
    
    # Grafico1

    # Dati
    L_fissa_mm = 5.0
    fattore_est_fisso = 1 + (L_fissa_mm / L_gain_mm)
    potenze_vett = np.logspace(-1, 2, 200)
    
    linewidth_Si3N4_vs_P = (linewidth_solitario / (fattore_est_fisso**2)) / potenze_vett + linewidth_floor
    effetto_TPA_vs_P = 15.0 * (potenze_vett / Soglia_TPA_Silicio_mW)**2.5
    linewidth_Silicio_vs_P = (linewidth_solitario / (fattore_est_fisso**2)) / potenze_vett + linewidth_floor + effetto_TPA_vs_P


    # Grafico2

    lunghezze_vett = np.linspace(0.0, 15.0, 200) # Da 0 a 15 mm di cavità passiva
    fattore_est_variabile = 1 + (lunghezze_vett / L_gain_mm)
    
    # Per il Nitruro di Silicio (Immune al TPA)
    linewidth_Si3N4_vs_L = (linewidth_solitario / (fattore_est_variabile**2)) / Potenza_Scelta_mW + linewidth_floor
    
    # Per il Silicio (Soffre di TPA in base alla potenza e rispetto alla sua soglia)
    effetto_TPA_vs_L = 15.0 * (Potenza_Scelta_mW / Soglia_TPA_Silicio_mW)**2.5
    linewidth_Silicio_vs_L = (linewidth_solitario / (fattore_est_variabile**2)) / Potenza_Scelta_mW + linewidth_floor + effetto_TPA_vs_L


    # Plotting

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Grafico1-plot

    ax1.loglog(potenze_vett, linewidth_Si3N4_vs_P, color='teal', lw=3, label='Nitruro di Silicio ($Si_3N_4$)')
    ax1.loglog(potenze_vett, linewidth_Silicio_vs_P, color='crimson', lw=2.5, linestyle='--', label='Silicio Classico (SOI)')
    ax1.axvline(Potenza_Scelta_mW, color='black', linestyle=':', alpha=0.8)
    ax1.axvline(Soglia_TPA_Silicio_mW, color='darkred', linestyle='--', alpha=0.4)
    ax1.set_title("Linewidth vs Potenza Ottica (Cavità fissa a 5 mm)", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Output Power (mW)", fontsize=10)
    ax1.set_ylabel("Intrinsic Linewidth (Hz)", fontsize=10)
    ax1.grid(True, which="both", linestyle=':', alpha=0.5)
    ax1.legend(loc='upper right')
    
    # Grafico2-plot

    # Usiamo la scala semilogaritmica sulle Y perché la lunghezza è lineare (0-15mm) ma la linewidth crolla di ordini di grandezza
    ax2.semilogy(lunghezze_vett, linewidth_Si3N4_vs_L, color='teal', lw=3, label='Nitruro di Silicio ($Si_3N_4$)')
    ax2.semilogy(lunghezze_vett, linewidth_Silicio_vs_L, color='crimson', lw=2.5, linestyle='--', label='Silicio Classico (SOI)')
    
    # Evidenziamo il punto a 0 mm (ovvero il diodo InP da solo senza circuito esterno)
    ax2.scatter(0, linewidth_solitario/Potenza_Scelta_mW, color='darkred', s=100, zorder=5, label='Solo Diodo InP (0 mm)')
    
    ax2.set_title(f"Linewidth vs Lunghezza Cavità Passiva (A {Potenza_Scelta_mW:.1f} mW)", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Feedback Cavity Length $L_f$ (mm)", fontsize=10)
    ax2.set_ylabel("Intrinsic Linewidth (Hz)", fontsize=10)
    ax2.grid(True, which="both", linestyle=':', alpha=0.5)
    ax2.legend(loc='upper right')
    
    # Vincoli di visualizzazione stabili
    ax1.set_ylim(10, 1e6)
    ax2.set_ylim(10, 1e7)
    
    plt.tight_layout()
    plt.show()

# Slider
interact(
    simulate_all_materials_and_geometry,
    Potenza_Scelta_mW=FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Potenza di lavoro (mW):', style={'description_width': 'initial'}, layout=Layout(width='60%')),
    Soglia_TPA_Silicio_mW=FloatSlider(value=10.0, min=5.0, max=30.0, step=1.0, description='Soglia TPA Silicio (mW):', style={'description_width': 'initial'}, layout=Layout(width='60%'))
);

interactive(children=(FloatSlider(value=15.0, description='Potenza di lavoro (mW):', layout=Layout(width='60%'…